# Multimodal Embedding Explorer

Advancing AI Anthropology through computational approaches to qualitative research.

[Matt Artz](https://www.mattartz.me/) | [GitHub](https://github.com/MattArtzAnthro) | [ORCID](https://orcid.org/0000-0002-3822-1429)

<br>

---

<br>

## What This Notebook Does

This notebook embeds ethnographic data — text documents, images, audio recordings, and video clips — into a unified vector space using Google's Gemini Embedding 2 model. Once embedded, you can explore relationships across your data through similarity analysis, clustering, interactive visualization, and a conversational search interface.

Gemini Embedding 2 is natively multimodal: it places text, images, audio, video, and PDF documents into the same embedding space, allowing direct comparison across media types. A field photo can be compared to a transcript passage, or an audio recording to a set of field notes.

**Privacy note**: Unlike the toolkit's local notebooks (Semantic Chunker, Audio Transcription), this notebook sends your data to Google's API to generate embeddings. Sending data to an API is a disclosure event — check your consent agreements before uploading identifiable interviews, images of participants, or other sensitive material.

## Key Features

- **Multimodal embedding**: Text, images, audio, video, and PDFs in a single vector space
- **No local GPU required**: Runs via Google's Gemini API — works on any Colab runtime
- **Automatic media handling**: Long audio is segmented, long PDFs are trimmed, and oversized files are rejected with guidance instead of failing at the API
- **Similarity analysis**: Pairwise cosine similarity with static and interactive heatmaps
- **Clustering**: HDBSCAN or KMeans grouping with automatic cluster characterization
- **2D/3D visualization**: UMAP projections with matplotlib (static) and plotly (interactive)
- **Cross-modal query**: Describe what you are looking for in words and retrieve the most similar items across every modality, with inline image and audio previews
- **Multi-format export**: Excel, CSV, JSON, NumPy, Parquet, and GEXF (Gephi) similarity networks

## Workflow

1. **Setup**: Install packages and configure your Gemini API key
2. **Configure**: Set embedding dimensions, task type, and project metadata
3. **Upload**: Load text, image, audio, and video files
4. **Embed**: Generate embeddings for all uploaded files via the Gemini API
5. **Explore**: Analyze similarity, cluster data, and visualize in 2D/3D
6. **Query**: Search your embedded data with natural language
7. **Export**: Download results in your preferred format

## Applications

This notebook supports research that involves working with mixed-media ethnographic data — comparing themes across interview transcripts and field photos, grouping audio recordings by content similarity, or building searchable collections of multimodal fieldwork data. Useful for dissertation research, applied projects, and any study where understanding relationships across different media types matters.

## Methodological Positioning

This notebook represents a **computational anthropology** approach — using vector embeddings to map relationships across heterogeneous data types. Embeddings capture statistical patterns in content, not cultural meaning. The similarity scores and clusters the notebook produces are starting points for interpretation, not conclusions.

**Important**: Embedding quality depends on the data. Short text snippets, low-quality audio, or heavily stylized images may produce less meaningful embeddings. Review results critically.

## Target Audience

Designed for anthropologists and qualitative researchers who want to explore relationships across multimodal fieldwork data — from graduate students organizing dissertation materials to applied researchers working with mixed-media collections.

## Technical Approach

The notebook uses **Gemini Embedding 2** (`gemini-embedding-2-preview`), Google's first natively multimodal embedding model. It maps text, images, audio, video, and documents into a unified 768–3,072 dimensional vector space via the Gemini API. Downstream analysis uses scikit-learn (similarity, clustering) and UMAP (dimensionality reduction). All computation runs in the notebook; only embedding generation requires the API. The model report is at https://arxiv.org/abs/2605.27295.

## AI Anthropology Toolkit

This notebook is part of the [AI Anthropology Toolkit](https://github.com/MattArtzAnthro/AI-Anthropology-Toolkit), a collection of computational notebooks for AI-assisted qualitative research. It pairs naturally with the analysis pipeline: transcripts from [Audio Transcription with Whisper](https://github.com/MattArtzAnthro/AI-Anthropology-Toolkit/blob/main/notebooks/Audio_Transcription_Whisper.ipynb), chunks from the [Interview Transcript Semantic Chunker](https://github.com/MattArtzAnthro/AI-Anthropology-Toolkit/blob/main/notebooks/Interview_Transcript_Semantic_Chunker.ipynb), and mixed fieldwork media can all be mapped in one space.

## License & Attribution

This notebook is released under the [PolyForm Noncommercial License 1.0.0](https://polyformproject.org/licenses/noncommercial/1.0.0): free for noncommercial use — research, education, and nonprofit work — with attribution to Matt Artz appreciated. For commercial licensing, contact [Matt Artz](https://www.mattartz.me/).

## Citation

If you use this toolkit in your academic research, please cite:

> Artz, Matt. 2026. AI Anthropology Toolkit. Software. Zenodo. https://doi.org/10.5281/zenodo.16728812

## References

Artz, Matt. 2023. From Machine Learning to Machine Knowing: A Digital Anthropology Approach for the Machine Interpretation of Cultures. UNESCO. https://unesdoc.unesco.org/ark:/48223/pf0000384902.

Artz, Matt. 2023. "Ten Predictions for AI and the Future of Anthropology." Anthropology News, May 8. https://doi.org/10.1111/AN.1605.

Artz, Matt. 2026. "Artificial Intelligence: The AI Anthropology Lifecycle (of, by, for AI)." In Practicing Digital Ethnography, edited by Devin Proctor. Routledge. https://doi.org/10.4324/9781032672663-29.

Artz, Matt. 2026. "Multi-Agent Ethnography: Post-Conventional Anthropological Practice Through Human-AI Collaboration." Human Organization. https://doi.org/10.1080/00664677.2026.2614501.

Artz, Matt. Forthcoming. "AI Anthropology: The Future of Applied Anthropological Practice." In Routledge Handbook of Applied Anthropology, edited by Christina Wasson, Edward B. Liebow, Karine L. Narahara, Ndukuyakhe Ndlovu, and Alaka Wali. New York: Routledge.

## Setup and Installation

Install required Python packages and import libraries. You will need a Gemini API key — get one free at [Google AI Studio](https://aistudio.google.com/apikey). Run this cell first.

In [ ]:
# Install required packages (numpy/pandas pinned for Colab compatibility)
!pip install -q google-genai pandas "numpy<2.2" matplotlib plotly scikit-learn umap-learn networkx openpyxl pypdf python-docx pyarrow pydub ipywidgets

import re
import json
import time
import unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')

from IPython.display import display, HTML, clear_output, Audio
import ipywidgets as widgets

# Google GenAI
from google import genai
from google.genai import types
print("✓ google-genai loaded")

# Plotly
import plotly.express as px
print("✓ plotly loaded")

# UMAP
import umap
print("✓ umap-learn loaded")

# PDF handling (page counts, trimming, text previews)
import pypdf
print("✓ pypdf loaded")

# DOCX text extraction
import docx
print("✓ python-docx loaded")

# Audio segmentation (uses ffmpeg, pre-installed on Colab)
from pydub import AudioSegment
print("✓ pydub loaded")

# HDBSCAN ships inside scikit-learn 1.3+ — no separate compiled package needed
HAVE_HDBSCAN = False
try:
    from sklearn.cluster import HDBSCAN as SklearnHDBSCAN
    HAVE_HDBSCAN = True
    print("✓ HDBSCAN available (scikit-learn)")
except ImportError:
    print("⚠️ HDBSCAN not available in this scikit-learn — will use KMeans for clustering")

HAVE_NETWORKX = False
try:
    import networkx as nx
    HAVE_NETWORKX = True
    print("✓ networkx loaded")
except ImportError:
    print("⚠️ networkx not available — GEXF export will be skipped")

# Colab detection
IN_COLAB = False
try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    pass

# Utilities
def make_slug(text):
    """Filesystem-safe slug from text."""
    text = unicodedata.normalize('NFKD', text)
    text = re.sub(r'[^\w\s-]', '', text).strip().lower()
    return re.sub(r'[-\s]+', '_', text)[:50] or 'project'

def normalize_text(text):
    """Normalize unicode artifacts in exported text."""
    if not isinstance(text, str):
        return text
    replacements = {
        '‑': '-', '‐': '-', '‒': '-', '–': '-', '—': '-',
        '‘': "'", '’': "'", '“': '"', '”': '"',
        '…': '...', '\xa0': ' ',
    }
    for k, v in replacements.items():
        text = text.replace(k, v)
    return text

def style_excel(filepath):
    """Apply branded styling to an Excel file."""
    from openpyxl import load_workbook
    from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
    wb = load_workbook(filepath)
    hf = PatternFill(start_color='274C77', end_color='274C77', fill_type='solid')
    hfont = Font(bold=True, color='FFFFFF')
    tb = Border(
        left=Side(style='thin', color='E7ECEF'), right=Side(style='thin', color='E7ECEF'),
        top=Side(style='thin', color='E7ECEF'), bottom=Side(style='thin', color='E7ECEF')
    )
    for ws in wb.worksheets:
        for cell in ws[1]:
            cell.fill = hf
            cell.font = hfont
            cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
            cell.border = tb
        for col in ws.columns:
            max_len = max(len(str(c.value or '')) for c in col)
            ws.column_dimensions[col[0].column_letter].width = min(max(max_len + 2, 10), 60)
        ws.freeze_panes = 'A2'
        for row in ws.iter_rows(min_row=2):
            for cell in row:
                cell.alignment = Alignment(vertical='top', wrap_text=True)
                cell.border = tb
    wb.save(filepath)

print()
print(f"✓ Environment: {'Google Colab' if IN_COLAB else 'Local Jupyter'}")
print("\U0001f4cb Ready to configure your embedding project!")


## Configuration

Set your Gemini API key, embedding parameters, and project metadata. The API key can be stored in Colab Secrets (recommended) or entered manually below.

In [ ]:
# Configuration

class EmbeddingConfig:
    """Configuration for multimodal embedding."""
    API_KEY = ''
    DIMENSIONS = 768
    TASK_TYPE = ''
    PROJECT_NAME = ''
    PROJECT_DESCRIPTION = ''

config = EmbeddingConfig()
client = None

# Try Colab secrets first
try:
    from google.colab import userdata
    config.API_KEY = userdata.get('GEMINI_API_KEY')
    print("\u2713 API key loaded from Colab Secrets")
except Exception:
    pass

style = {'description_width': '160px'}
layout = widgets.Layout(width='500px')

def create_config_interface():
    global client

    config_html = '<div style="background-color: #E7ECEF; padding: 20px; border-radius: 10px; margin: 20px 0; border-left: 5px solid #274C77;">'
    config_html += '<h2 style="color: #274C77; margin-top: 0;">\U0001f9ec Multimodal Embedding Explorer</h2>'
    config_html += '<p><strong>Welcome!</strong> This notebook embeds ethnographic data (text, images, audio, video) into a unified vector space using Gemini Embedding 2.</p>'
    config_html += '<h3 style="color: #274C77;">\U0001f4d6 How to Use:</h3>'
    config_html += '<ol>'
    config_html += '<li><strong>Configure:</strong> Set your API key and parameters below</li>'
    config_html += '<li><strong>Upload:</strong> Load text, images, audio, or video files</li>'
    config_html += '<li><strong>Embed:</strong> Generate embeddings via the Gemini API</li>'
    config_html += '<li><strong>Explore:</strong> Analyze similarity, clusters, and visualizations</li>'
    config_html += '<li><strong>Chat:</strong> Query your data through a conversational interface</li>'
    config_html += '<li><strong>Export:</strong> Download results in your preferred format</li>'
    config_html += '</ol>'
    config_html += '</div>'

    instructions = widgets.HTML(value=config_html)

    api_header = widgets.HTML('<h3 style="margin: 20px 0 10px 0; color: #274C77;">\U0001f511 API Key</h3>')

    api_key_input = widgets.Password(
        value=config.API_KEY,
        placeholder='Enter your Gemini API key',
        description='Gemini API Key:',
        layout=layout,
        style=style
    )

    api_help = widgets.HTML(
        '<p style="color: #8B8C89; font-size: 0.85em; margin: 2px 0 10px 0;">'
        'Get a free API key at <a href="https://aistudio.google.com/apikey" target="_blank">Google AI Studio</a>. '
        'In Colab, you can also store it in Secrets (key icon in sidebar) as GEMINI_API_KEY.</p>'
    )

    embed_header = widgets.HTML('<h3 style="margin: 20px 0 10px 0; color: #274C77;">\u2699\ufe0f Embedding Parameters</h3>')

    dim_input = widgets.Dropdown(
        options=[
            ('768 dimensions (recommended)', 768),
            ('1,536 dimensions', 1536),
            ('3,072 dimensions (maximum)', 3072),
        ],
        value=768,
        description='Dimensions:',
        layout=layout,
        style=style
    )

    dim_help = widgets.HTML(
        '<p style="color: #8B8C89; font-size: 0.85em; margin: 2px 0 10px 0;">'
        '768 offers the best balance of quality and file size. Higher dimensions increase precision slightly but produce larger exports.</p>'
    )

    task_input = widgets.Dropdown(
        options=[
            ('Model default (recommended)', ''),
            ('Semantic Similarity', 'SEMANTIC_SIMILARITY'),
            ('Clustering', 'CLUSTERING'),
            ('Classification', 'CLASSIFICATION'),
            ('Document Retrieval', 'RETRIEVAL_DOCUMENT'),
            ('Query Retrieval', 'RETRIEVAL_QUERY'),
            ('Question Answering', 'QUESTION_ANSWERING'),
        ],
        value='',
        description='Task type:',
        layout=layout,
        style=style
    )

    task_help = widgets.HTML(
        '<p style="color: #8B8C89; font-size: 0.85em; margin: 2px 0 10px 0;">'
        'Optional optimization hint. The preview API does not accept task types for every input; '
        'when a call is rejected, the notebook automatically retries without it. Model default is the safest choice.</p>'
    )

    project_header = widgets.HTML('<h3 style="margin: 20px 0 10px 0; color: #274C77;">\U0001f4c1 Project Metadata</h3>')

    project_name = widgets.Text(
        value='',
        placeholder='e.g., Market Ethnography 2026',
        description='Project name:',
        layout=layout,
        style=style
    )

    project_desc = widgets.Textarea(
        value='',
        placeholder='Brief description of the data and research context',
        description='Description:',
        layout=widgets.Layout(width='500px', height='80px'),
        style=style
    )

    save_btn = widgets.Button(
        description='Save Configuration',
        style={'button_color': '#6096BA'},
        layout=widgets.Layout(width='200px', margin='20px 0')
    )

    status = widgets.Output()

    def save_config(btn):
        global client
        config.API_KEY = api_key_input.value.strip()
        config.DIMENSIONS = dim_input.value
        config.TASK_TYPE = task_input.value
        config.PROJECT_NAME = project_name.value.strip()
        config.PROJECT_DESCRIPTION = project_desc.value.strip()

        with status:
            clear_output()
            if not config.API_KEY:
                print("\u26a0\ufe0f Please enter a Gemini API key.")
                return

            print("Validating API key...")
            try:
                client = genai.Client(api_key=config.API_KEY)
                test = client.models.embed_content(
                    model='gemini-embedding-2-preview',
                    contents='test',
                    config=types.EmbedContentConfig(output_dimensionality=256)
                )
                print("\u2713 API key validated \u2014 Gemini Embedding 2 is accessible")
                print(f"\u2713 Dimensions: {config.DIMENSIONS}")
                print(f"\u2713 Task type: {config.TASK_TYPE or 'model default'}")
                if config.PROJECT_NAME:
                    print(f"\u2713 Project: {config.PROJECT_NAME}")
                print()
                print("\u2713 Configuration saved! Proceed to the Data Upload cell.")
            except Exception as e:
                print(f"\u274c API validation failed: {e}")
                client = None

    save_btn.on_click(save_config)

    display(widgets.VBox([
        instructions,
        api_header,
        api_key_input,
        api_help,
        embed_header,
        dim_input,
        dim_help,
        task_input,
        task_help,
        project_header,
        project_name,
        project_desc,
        save_btn,
        status,
    ]))

create_config_interface()

## Data Upload

Upload your ethnographic data files. The notebook accepts text (.txt, .docx), PDF documents, images (.png, .jpg), audio (.mp3, .wav), and video (.mp4, .mov). Select multiple files at once or upload in batches.

Files that exceed the API's media limits are handled automatically: audio over 75 seconds is split into segments, PDFs over 6 pages are trimmed, and oversized or over-length files are rejected with clear guidance.

In [ ]:
# Data Upload

MODALITY_MAP = {
    '.txt':  ('text',     'text/plain'),
    '.pdf':  ('document', 'application/pdf'),
    '.docx': ('text',     'text/plain'),
    '.png':  ('image',    'image/png'),
    '.jpg':  ('image',    'image/jpeg'),
    '.jpeg': ('image',    'image/jpeg'),
    '.mp3':  ('audio',    'audio/mpeg'),
    '.wav':  ('audio',    'audio/wav'),
    '.mp4':  ('video',    'video/mp4'),
    '.mov':  ('video',    'video/quicktime'),
}

# API limits for gemini-embedding-2-preview, with a safety margin.
# Oversized media fails at the API with unhelpful errors, so it is
# checked (and where possible, fixed) here instead.
MAX_AUDIO_SECONDS = 75     # API limit: 80s — longer audio is segmented
MAX_VIDEO_SECONDS = 125    # API limit: 128s — longer video is rejected with guidance
MAX_PDF_PAGES = 6          # API limit — longer PDFs are trimmed to the first 6 pages
MAX_INLINE_MB = 15         # inline request ceiling — larger files are rejected

uploaded_items = []

def extract_text_from_pdf(raw_bytes):
    """Extract text from PDF bytes for preview."""
    import io
    reader = pypdf.PdfReader(io.BytesIO(raw_bytes))
    pages = []
    for page in reader.pages:
        text = page.extract_text()
        if text:
            pages.append(text)
    return '\n'.join(pages)

def extract_text_from_docx(raw_bytes):
    """Extract text from DOCX bytes."""
    import io
    doc_obj = docx.Document(io.BytesIO(raw_bytes))
    return '\n'.join(p.text for p in doc_obj.paragraphs if p.text.strip())

def trim_pdf(raw_bytes, max_pages):
    """Return (bytes, n_pages, trimmed) keeping at most max_pages pages."""
    import io
    reader = pypdf.PdfReader(io.BytesIO(raw_bytes))
    n_pages = len(reader.pages)
    if n_pages <= max_pages:
        return raw_bytes, n_pages, False
    writer = pypdf.PdfWriter()
    for page in reader.pages[:max_pages]:
        writer.add_page(page)
    buf = io.BytesIO()
    writer.write(buf)
    return buf.getvalue(), n_pages, True

def segment_audio(raw_bytes, ext, max_seconds):
    """Split audio into <= max_seconds segments. Returns list of
    (segment_bytes, mime_type, start_s, end_s)."""
    import io
    audio = AudioSegment.from_file(io.BytesIO(raw_bytes), format=ext.lstrip('.'))
    duration_s = len(audio) / 1000.0
    segments = []
    step_ms = max_seconds * 1000
    for start_ms in range(0, len(audio), step_ms):
        piece = audio[start_ms:start_ms + step_ms]
        buf = io.BytesIO()
        piece.export(buf, format='mp3')
        segments.append((buf.getvalue(), 'audio/mpeg',
                         start_ms / 1000.0, min((start_ms + step_ms) / 1000.0, duration_s)))
    return segments

def media_duration_seconds(raw_bytes, ext):
    """Duration of an audio/video file via pydub/ffmpeg. None if unreadable."""
    import io
    try:
        seg = AudioSegment.from_file(io.BytesIO(raw_bytes), format=ext.lstrip('.'))
        return len(seg) / 1000.0
    except Exception:
        return None

def preflight_items(fname, ext, raw_bytes, notes):
    """Validate one upload against API limits. Returns a list of item dicts
    (an oversize audio file becomes several segment items), or [] if rejected."""
    modality, mime_type = MODALITY_MAP[ext]
    size_mb = len(raw_bytes) / (1024 * 1024)

    if size_mb > MAX_INLINE_MB and modality != 'audio':
        notes.append(f"❌ {fname}: {size_mb:.1f} MB exceeds the {MAX_INLINE_MB} MB "
                     "request ceiling — compress or trim the file and re-upload.")
        return []

    text_content = None
    if ext == '.txt':
        try:
            text_content = raw_bytes.decode('utf-8')
        except UnicodeDecodeError:
            text_content = raw_bytes.decode('latin-1')
    elif ext == '.docx':
        try:
            text_content = extract_text_from_docx(raw_bytes)
        except Exception as e:
            notes.append(f"⚠️ {fname}: could not extract text ({e}) — skipped.")
            return []
    elif ext == '.pdf':
        try:
            raw_bytes, n_pages, trimmed = trim_pdf(raw_bytes, MAX_PDF_PAGES)
            if trimmed:
                notes.append(f"✂️ {fname}: {n_pages} pages — trimmed to the first "
                             f"{MAX_PDF_PAGES} (API limit). Split the PDF and re-upload "
                             "the rest if you need every page.")
            text_content = extract_text_from_pdf(raw_bytes)
        except Exception:
            text_content = ''
    elif modality == 'audio':
        duration = media_duration_seconds(raw_bytes, ext)
        if duration is None:
            notes.append(f"⚠️ {fname}: could not read audio — skipped.")
            return []
        if duration > MAX_AUDIO_SECONDS:
            pieces = segment_audio(raw_bytes, ext, MAX_AUDIO_SECONDS)
            notes.append(f"✂️ {fname}: {duration:.0f}s — split into {len(pieces)} "
                         f"segments of ≤{MAX_AUDIO_SECONDS}s (API limit is 80s). "
                         "Each segment becomes its own point on the map.")
            items = []
            for i, (seg_bytes, seg_mime, start_s, end_s) in enumerate(pieces, 1):
                seg_mb = len(seg_bytes) / (1024 * 1024)
                if seg_mb > MAX_INLINE_MB:
                    notes.append(f"❌ {fname} [part {i}]: segment still {seg_mb:.1f} MB — skipped.")
                    continue
                items.append({
                    'filename': f"{fname} [part {i}/{len(pieces)}]",
                    'source_file': fname,
                    'segment_start_s': round(start_s, 1),
                    'segment_end_s': round(end_s, 1),
                    'modality': 'audio',
                    'mime_type': seg_mime,
                    'raw_bytes': seg_bytes,
                    'text_content': None,
                    'file_size_kb': round(len(seg_bytes) / 1024, 1),
                })
            return items
    elif modality == 'video':
        duration = media_duration_seconds(raw_bytes, ext)
        if duration is not None and duration > MAX_VIDEO_SECONDS:
            notes.append(f"❌ {fname}: {duration:.0f}s exceeds the {MAX_VIDEO_SECONDS}s "
                         "video limit — trim it, or run it through the Audio "
                         "Transcription with Whisper notebook and embed the transcript.")
            return []

    if modality == 'text' and not (text_content or '').strip():
        notes.append(f"⚠️ {fname}: no readable text found — skipped.")
        return []

    return [{
        'filename': fname,
        'source_file': fname,
        'modality': modality,
        'mime_type': mime_type,
        'raw_bytes': raw_bytes,
        'text_content': text_content,
        'file_size_kb': round(len(raw_bytes) / 1024, 1),
    }]

def process_uploaded_files(change):
    """Process files from the upload widget."""
    upload_out.clear_output()
    new_items = []
    notes = []

    for file_info in uploader.value:
        fname = file_info['name'] if isinstance(file_info, dict) else file_info.name
        content = file_info['content'] if isinstance(file_info, dict) else file_info.content
        raw_bytes = bytes(content)

        ext = '.' + fname.rsplit('.', 1)[-1].lower() if '.' in fname else ''
        if ext not in MODALITY_MAP:
            notes.append(f"⚠️ Skipping {fname} — unsupported format")
            continue

        if any(u['source_file'] == fname for u in uploaded_items):
            continue

        try:
            new_items.extend(preflight_items(fname, ext, raw_bytes, notes))
        except Exception as e:
            notes.append(f"⚠️ Error processing {fname}: {e}")

    uploaded_items.extend(new_items)

    with upload_out:
        for note in notes:
            print(note)
        if new_items:
            print(f"✓ Added {len(new_items)} item(s) ({len(uploaded_items)} total)")
            print()
            summary = pd.DataFrame([
                {'Item': u['filename'], 'Modality': u['modality'], 'Size (KB)': u['file_size_kb']}
                for u in uploaded_items
            ])
            display(summary)

            print()
            modalities = {}
            for u in uploaded_items:
                modalities[u['modality']] = modalities.get(u['modality'], 0) + 1
            for mod, count in sorted(modalities.items()):
                print(f"   {mod}: {count} item(s)")

upload_header_html = '<div style="background-color: #E7ECEF; padding: 15px; border-radius: 10px; margin: 10px 0; border-left: 5px solid #274C77;">'
upload_header_html += '<h3 style="color: #274C77; margin-top: 0;">\U0001f4c2 Upload Files</h3>'
upload_header_html += '<p>Supported formats and how limits are handled:</p>'
upload_header_html += '<ul style="margin: 5px 0;">'
upload_header_html += '<li><strong>Text:</strong> .txt, .docx</li>'
upload_header_html += '<li><strong>Documents:</strong> .pdf — trimmed to the first 6 pages if longer (API limit)</li>'
upload_header_html += '<li><strong>Images:</strong> .png, .jpg</li>'
upload_header_html += '<li><strong>Audio:</strong> .mp3, .wav — recordings over 75s are split into segments automatically</li>'
upload_header_html += '<li><strong>Video:</strong> .mp4, .mov — up to ~2 minutes; longer videos are rejected with guidance</li>'
upload_header_html += '</ul>'
upload_header_html += '<p style="color: #8B8C89; font-size: 0.9em;">Uploaded data is sent to the Gemini API for embedding — see the privacy note in the introduction before uploading sensitive material.</p>'
upload_header_html += '</div>'

upload_instructions = widgets.HTML(value=upload_header_html)

uploader = widgets.FileUpload(
    accept='.txt,.pdf,.docx,.png,.jpg,.jpeg,.mp3,.wav,.mp4,.mov',
    multiple=True,
    description='Select files',
    layout=widgets.Layout(width='300px')
)

uploader.observe(process_uploaded_files, names='value')

clear_btn = widgets.Button(
    description='Clear All Files',
    style={'button_color': '#8B8C89'},
    layout=widgets.Layout(width='150px', margin='10px 0')
)

upload_out = widgets.Output()

def on_clear(_):
    uploaded_items.clear()
    upload_out.clear_output()
    with upload_out:
        print("✓ All files cleared.")

clear_btn.on_click(on_clear)

display(widgets.VBox([
    upload_instructions,
    uploader,
    clear_btn,
    upload_out,
]))


## Generate Embeddings

Embed all uploaded files via the Gemini API. Each file produces a single embedding vector. Processing time depends on file count and size — the notebook includes rate limiting and retry logic.

In [ ]:
# Generate Embeddings

embeddings_df = None
embedding_matrix = None
embedded_results = {}   # filename -> record; survives re-runs so retries skip finished work
CHECKPOINT_PATH = 'embeddings_checkpoint.parquet'

def _embed_call(client, item, config, use_task_type):
    """One embed_content call. use_task_type toggles the task_type parameter."""
    kwargs = {'output_dimensionality': config.DIMENSIONS}
    if use_task_type and config.TASK_TYPE:
        kwargs['task_type'] = config.TASK_TYPE
    embed_config = types.EmbedContentConfig(**kwargs)

    if item['modality'] == 'text':
        contents = item['text_content']
    else:
        contents = [types.Part.from_bytes(
            data=item['raw_bytes'],
            mime_type=item['mime_type']
        )]
    result = client.models.embed_content(
        model='gemini-embedding-2-preview',
        contents=contents,
        config=embed_config
    )
    return result.embeddings[0].values

def embed_item(client, item, config):
    """Embed one item. If the API rejects the task_type parameter, retry
    without it — the preview model does not accept it for every input."""
    if not config.TASK_TYPE:
        return _embed_call(client, item, config, use_task_type=False)
    try:
        return _embed_call(client, item, config, use_task_type=True)
    except Exception as e:
        msg = str(e).lower()
        if 'task' in msg or 'invalid' in msg or '400' in msg:
            return _embed_call(client, item, config, use_task_type=False)
        raise

def _retry_wait(error_text, attempt):
    """Backoff seconds: honor a server-suggested retry delay when present."""
    match = re.search(r'retry(?:_delay)?\D{0,12}(\d+(?:\.\d+)?)', error_text.lower())
    if match:
        return min(float(match.group(1)) + 1, 60)
    return min(2 * (2 ** attempt), 45)

def embed_with_retry(client, item, config, max_retries=5):
    """Embed with exponential backoff on rate-limit and transient errors."""
    for attempt in range(max_retries + 1):
        try:
            return embed_item(client, item, config)
        except Exception as e:
            error_str = str(e).lower()
            transient = ('429' in error_str or 'rate' in error_str
                         or '503' in error_str or 'deadline' in error_str
                         or 'unavailable' in error_str)
            if transient and attempt < max_retries:
                time.sleep(_retry_wait(error_str, attempt))
                continue
            raise

def _save_checkpoint():
    """Persist finished embeddings (metadata + vectors, no raw bytes)."""
    try:
        pd.DataFrame(list(embedded_results.values())).to_parquet(CHECKPOINT_PATH, index=False)
    except Exception:
        pass  # checkpointing must never break the run

embed_btn = widgets.Button(
    description='Generate Embeddings',
    style={'button_color': '#6096BA'},
    layout=widgets.Layout(width='200px', margin='10px 0')
)

progress_html = widgets.HTML(value='')
embed_out = widgets.Output()

def on_embed(_):
    global embeddings_df, embedding_matrix
    embed_out.clear_output()
    progress_html.value = ''

    if client is None:
        with embed_out:
            print("⚠️ No API client — run the Configuration cell first.")
        return

    if not uploaded_items:
        with embed_out:
            print("⚠️ No files uploaded — run the Data Upload cell first.")
        return

    pending = [it for it in uploaded_items if it['filename'] not in embedded_results]
    total = len(uploaded_items)
    failures = []
    call_delay = 1.5

    with embed_out:
        print(f"\U0001f9ec Embedding {len(pending)} of {total} item(s) with Gemini Embedding 2...")
        if len(pending) < total:
            print(f"   ({total - len(pending)} already embedded — click again after failures to resume)")
        print(f"   Dimensions: {config.DIMENSIONS}")
        print(f"   Task type: {config.TASK_TYPE or 'model default'}")
        print()

    for idx, item in enumerate(pending):
        progress_html.value = (
            f'<span style="color: #274C77;">Embedding item {idx + 1}/{len(pending)} — '
            f'{item["filename"]} ({item["modality"]})</span>'
        )

        try:
            values = embed_with_retry(client, item, config)
            vec = np.array(values)

            # MRL: re-normalize truncated dimensions
            if config.DIMENSIONS != 3072:
                norm = np.linalg.norm(vec)
                if norm > 0:
                    vec = vec / norm

            preview = 'N/A'
            if item['text_content']:
                preview = normalize_text(item['text_content'][:300])

            embedded_results[item['filename']] = {
                'filename': item['filename'],
                'source_file': item.get('source_file', item['filename']),
                'modality': item['modality'],
                'mime_type': item['mime_type'],
                'file_size_kb': item['file_size_kb'],
                'text_preview': preview,
                'embedding': vec.tolist(),
            }

            if (idx + 1) % 10 == 0:
                _save_checkpoint()

        except Exception as e:
            failures.append((item['filename'], str(e)))
            if '429' in str(e) or 'rate' in str(e).lower():
                call_delay = max(call_delay, 5.0)
                with embed_out:
                    print(f"   ⏳ Rate limited — slowing to {call_delay:.0f}s between calls")

        if idx < len(pending) - 1:
            time.sleep(call_delay)

    progress_html.value = ''
    _save_checkpoint()

    if not embedded_results:
        with embed_out:
            print("❌ No items were successfully embedded.")
            for fname, err in failures:
                print(f"   {fname}: {err}")
        return

    ordered = [embedded_results[it['filename']] for it in uploaded_items
               if it['filename'] in embedded_results]
    embeddings_df = pd.DataFrame(ordered)
    embedding_matrix = np.array(embeddings_df['embedding'].tolist())

    with embed_out:
        print(f"✓ Embedded {len(embeddings_df)}/{total} item(s)", end='')
        if failures:
            print(f" ({len(failures)} failed — click Generate Embeddings again to retry just those)")
            for fname, err in failures:
                print(f"   ⚠️ {fname}: {err[:200]}")
        else:
            print()

        print(f"   Matrix shape: {embedding_matrix.shape}")
        print(f"   Checkpoint saved to {CHECKPOINT_PATH}")
        print()
        print("Proceed to the Exploration cells.")

embed_btn.on_click(on_embed)
display(widgets.VBox([embed_btn, progress_html, embed_out]))


## Similarity Analysis

Compute pairwise cosine similarity between all embedded items. The heatmap shows how similar each pair of files is — warmer colors indicate higher similarity. Items from different modalities (e.g., a transcript and a photo) can have meaningful similarity because they share the same embedding space.

In [ ]:
# Similarity Analysis

if embedding_matrix is None:
    print("\u26a0\ufe0f Run the embedding cell first.")
else:
    # Compute similarity matrix
    sim_matrix = cosine_similarity(embedding_matrix)
    filenames = embeddings_df['filename'].tolist()
    n = len(filenames)

    sim_df = pd.DataFrame(sim_matrix, index=filenames, columns=filenames)

    # --- Static heatmap (matplotlib) ---
    fig, ax = plt.subplots(figsize=(max(8, n * 0.6), max(6, n * 0.5)))
    fig.patch.set_facecolor('#FFFFFF')

    from matplotlib.colors import LinearSegmentedColormap
    brand_cmap = LinearSegmentedColormap.from_list('brand', ['#E7ECEF', '#A3CEF1', '#6096BA', '#274C77'])

    im = ax.imshow(sim_matrix, cmap=brand_cmap, vmin=0, vmax=1, aspect='auto')

    ax.set_xticks(range(n))
    ax.set_yticks(range(n))
    truncated = [f[:20] + '...' if len(f) > 20 else f for f in filenames]
    ax.set_xticklabels(truncated, rotation=45, ha='right', fontsize=9)
    ax.set_yticklabels(truncated, fontsize=9)

    # Annotate if small enough
    if n <= 20:
        for i in range(n):
            for j in range(n):
                color = 'white' if sim_matrix[i, j] > 0.6 else '#274C77'
                ax.text(j, i, f'{sim_matrix[i, j]:.2f}', ha='center', va='center',
                        fontsize=7, color=color)

    ax.set_title('Pairwise Cosine Similarity', fontsize=14, color='#274C77', fontweight='bold', pad=15)
    fig.colorbar(im, ax=ax, shrink=0.8, label='Similarity')

    plt.tight_layout()
    plt.show()

    # --- Interactive heatmap (plotly) ---
    fig_px = px.imshow(
        sim_matrix,
        x=filenames,
        y=filenames,
        color_continuous_scale=['#E7ECEF', '#A3CEF1', '#6096BA', '#274C77'],
        zmin=0, zmax=1,
        labels={'color': 'Similarity'},
        title='Pairwise Cosine Similarity (Interactive)',
    )
    fig_px.update_layout(width=max(600, n * 50), height=max(500, n * 45))
    fig_px.show()

    # --- Top/bottom pairs ---
    print()
    pairs = []
    for i in range(n):
        for j in range(i + 1, n):
            pairs.append((filenames[i], filenames[j], sim_matrix[i, j]))
    pairs.sort(key=lambda x: x[2], reverse=True)

    print("\U0001f50d Top 5 most similar pairs:")
    for a, b, s in pairs[:5]:
        print(f"   {s:.3f}  {a} \u2194 {b}")

    print()
    print("\U0001f50e Top 5 least similar pairs:")
    for a, b, s in pairs[-5:]:
        print(f"   {s:.3f}  {a} \u2194 {b}")

## Clustering

Group embedded items by similarity using density-based (HDBSCAN) or centroid-based (KMeans) clustering. Clusters can reveal thematic groupings across different media types.

In [ ]:
# Clustering

if embedding_matrix is None:
    print("\u26a0\ufe0f Run the embedding cell first.")
elif len(embedding_matrix) < 3:
    print("\u26a0\ufe0f Clustering requires at least 3 embedded items.")
else:
    n_items = len(embedding_matrix)

    cluster_labels = None

    if HAVE_HDBSCAN:
        print("\U0001f50d Running HDBSCAN clustering...")
        min_size = min(3, n_items)
        clusterer = SklearnHDBSCAN(min_cluster_size=min_size, metric='euclidean')
        cluster_labels = clusterer.fit_predict(embedding_matrix)

        n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
        n_noise = (cluster_labels == -1).sum()

        if n_clusters == 0:
            print(f"   HDBSCAN found no clusters (all {n_noise} items marked as noise). Falling back to KMeans.")
            cluster_labels = None
        else:
            print(f"   \u2713 Found {n_clusters} clusters ({n_noise} noise points)")

    if cluster_labels is None:
        # KMeans fallback
        print("\U0001f50d Running KMeans clustering...")

        max_k = min(8, n_items - 1)
        if max_k < 2:
            print("   Not enough items for KMeans (need at least 3).")
        else:
            best_k = 2
            best_score = -1

            for k in range(2, max_k + 1):
                km = KMeans(n_clusters=k, random_state=42, n_init=10)
                labels = km.fit_predict(embedding_matrix)
                if len(set(labels)) > 1:
                    score = silhouette_score(embedding_matrix, labels)
                    if score > best_score:
                        best_score = score
                        best_k = k

            km = KMeans(n_clusters=best_k, random_state=42, n_init=10)
            cluster_labels = km.fit_predict(embedding_matrix)
            print(f"   \u2713 KMeans selected k={best_k} (silhouette: {best_score:.3f})")

    if cluster_labels is not None:
        embeddings_df['cluster'] = cluster_labels

        # Cluster summary
        print()
        print("\U0001f4ca Cluster Summary:")
        for label in sorted(set(cluster_labels)):
            if label == -1:
                label_name = "Noise"
            else:
                label_name = f"Cluster {label}"

            mask = cluster_labels == label
            members = embeddings_df[mask]
            modalities = members['modality'].value_counts()
            mod_str = ', '.join(f'{m}: {c}' for m, c in modalities.items())

            print(f"\n   {label_name} ({mask.sum()} items \u2014 {mod_str}):")
            for _, row in members.iterrows():
                print(f"      - {row['filename']} ({row['modality']})")

## Visualization

Project embeddings into 2D and 3D space using UMAP for visual exploration. Points are colored by modality and shaped by cluster. The interactive plots allow hovering to see file details.

In [ ]:
# Visualization

MODALITY_COLORS = {
    'text': '#274C77',
    'document': '#3A6B97',
    'image': '#6096BA',
    'audio': '#A3CEF1',
    'video': '#8B8C89',
}

CLUSTER_MARKERS_MPL = ['o', 's', '^', 'D', 'v', 'P', '*', 'X']

if embedding_matrix is None:
    print("\u26a0\ufe0f Run the embedding cell first.")
elif len(embedding_matrix) < 5:
    print(f"\u26a0\ufe0f UMAP visualization requires at least 5 embedded items. You have {len(embedding_matrix)}.")
    print("   Skipping visualization. Similarity matrix and clustering still work.")
else:
    n_items = len(embedding_matrix)
    n_neighbors = min(15, n_items - 1)

    print(f"\U0001f4ca Running UMAP (n_neighbors={n_neighbors})...")

    # 2D UMAP
    reducer_2d = umap.UMAP(n_components=2, n_neighbors=n_neighbors, random_state=42)
    coords_2d = reducer_2d.fit_transform(embedding_matrix)

    # 3D UMAP
    reducer_3d = umap.UMAP(n_components=3, n_neighbors=n_neighbors, random_state=42)
    coords_3d = reducer_3d.fit_transform(embedding_matrix)

    embeddings_df['umap_x'] = coords_2d[:, 0]
    embeddings_df['umap_y'] = coords_2d[:, 1]
    embeddings_df['umap_3d_x'] = coords_3d[:, 0]
    embeddings_df['umap_3d_y'] = coords_3d[:, 1]
    embeddings_df['umap_3d_z'] = coords_3d[:, 2]

    has_clusters = 'cluster' in embeddings_df.columns

    # --- Static 2D (matplotlib) ---
    fig, ax = plt.subplots(figsize=(10, 8))
    fig.patch.set_facecolor('#FFFFFF')
    ax.set_facecolor('#FAFAFA')

    for _, row in embeddings_df.iterrows():
        color = MODALITY_COLORS.get(row['modality'], '#8B8C89')
        marker = 'o'
        if has_clusters:
            cl = int(row['cluster']) if row['cluster'] != -1 else 0
            marker = CLUSTER_MARKERS_MPL[cl % len(CLUSTER_MARKERS_MPL)]
        ax.scatter(row['umap_x'], row['umap_y'], c=color, marker=marker, s=100,
                   edgecolors='white', linewidth=0.5, zorder=3)

        label = row['filename'][:15] + '...' if len(row['filename']) > 15 else row['filename']
        ax.annotate(label, (row['umap_x'], row['umap_y']), fontsize=7, ha='left',
                    xytext=(5, 5), textcoords='offset points', color='#274C77')

    ax.set_title('Embedding Space (2D UMAP)', fontsize=14, color='#274C77', fontweight='bold', pad=15)
    ax.set_xlabel('UMAP 1', fontsize=11, color='#274C77')
    ax.set_ylabel('UMAP 2', fontsize=11, color='#274C77')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color('#E7ECEF')
    ax.spines['bottom'].set_color('#E7ECEF')
    ax.grid(True, alpha=0.2, color='#8B8C89')

    # Legend for modalities
    from matplotlib.lines import Line2D
    legend_elements = [Line2D([0], [0], marker='o', color='w', markerfacecolor=c, markersize=10, label=m)
                       for m, c in MODALITY_COLORS.items() if m in embeddings_df['modality'].values]
    ax.legend(handles=legend_elements, title='Modality', frameon=True, facecolor='white', edgecolor='#E7ECEF')

    plt.tight_layout()
    plt.show()

    # --- Interactive 2D (plotly) ---
    hover_data = {
        'filename': True,
        'modality': True,
        'file_size_kb': True,
        'umap_x': False,
        'umap_y': False,
    }
    if has_clusters:
        hover_data['cluster'] = True

    # Truncate previews for hover
    embeddings_df['hover_preview'] = embeddings_df['text_preview'].apply(
        lambda x: x[:100] + '...' if isinstance(x, str) and x != 'N/A' and len(x) > 100 else x
    )
    hover_data['hover_preview'] = True

    symbol_col = 'cluster' if has_clusters else None

    fig_2d = px.scatter(
        embeddings_df,
        x='umap_x', y='umap_y',
        color='modality',
        color_discrete_map=MODALITY_COLORS,
        symbol=symbol_col,
        hover_data=hover_data,
        title='Embedding Space (2D UMAP \u2014 Interactive)',
        labels={'umap_x': 'UMAP 1', 'umap_y': 'UMAP 2'},
    )
    fig_2d.update_traces(marker=dict(size=12, line=dict(width=1, color='white')))
    fig_2d.update_layout(width=800, height=600)
    fig_2d.show()

    # --- Interactive 3D (plotly) ---
    fig_3d = px.scatter_3d(
        embeddings_df,
        x='umap_3d_x', y='umap_3d_y', z='umap_3d_z',
        color='modality',
        color_discrete_map=MODALITY_COLORS,
        symbol=symbol_col,
        hover_data=hover_data,
        title='Embedding Space (3D UMAP \u2014 Interactive)',
        labels={'umap_3d_x': 'UMAP 1', 'umap_3d_y': 'UMAP 2', 'umap_3d_z': 'UMAP 3'},
    )
    fig_3d.update_traces(marker=dict(size=6, line=dict(width=0.5, color='white')))
    fig_3d.update_layout(width=800, height=700)
    fig_3d.show()

    # Clean up temp column
    if 'hover_preview' in embeddings_df.columns:
        embeddings_df.drop(columns=['hover_preview'], inplace=True)

    print("\u2713 Visualization complete")

## Query Your Data

Query your embedded ethnographic data using natural language. Enter a question or description and the notebook finds the most similar items across all modalities. This works because your query is embedded into the same vector space as your data.

In [ ]:
# Query Your Data

if embedding_matrix is None:
    print("⚠️ Run the embedding cell first.")
else:
    query_input = widgets.Text(
        value='',
        placeholder='e.g., conversations about market bargaining',
        description='Query:',
        style={'description_width': '60px'},
        layout=widgets.Layout(width='500px')
    )

    topk_slider = widgets.IntSlider(
        value=5, min=1, max=min(20, len(embedding_matrix)),
        description='Results:',
        style={'description_width': '60px'},
        layout=widgets.Layout(width='400px')
    )

    search_btn = widgets.Button(
        description='Search',
        style={'button_color': '#6096BA'},
        layout=widgets.Layout(width='150px', margin='10px 0')
    )

    query_out = widgets.Output()

    def _embed_query(text):
        """Embed a text query into the same space as the data."""
        query_task = config.TASK_TYPE
        if config.TASK_TYPE == 'RETRIEVAL_DOCUMENT':
            query_task = 'RETRIEVAL_QUERY'
        kwargs = {'output_dimensionality': config.DIMENSIONS}
        if query_task:
            kwargs['task_type'] = query_task
        try:
            result = client.models.embed_content(
                model='gemini-embedding-2-preview',
                contents=text,
                config=types.EmbedContentConfig(**kwargs)
            )
        except Exception:
            result = client.models.embed_content(
                model='gemini-embedding-2-preview',
                contents=text,
                config=types.EmbedContentConfig(output_dimensionality=config.DIMENSIONS)
            )
        vec = np.array(result.embeddings[0].values)
        norm = np.linalg.norm(vec)
        return vec / norm if norm > 0 else vec

    def _preview_widget(row):
        """Inline preview for a result: thumbnail for images, player for audio."""
        source = next((u for u in uploaded_items if u['filename'] == row['filename']), None)
        if source is None:
            return None
        if row['modality'] == 'image':
            return widgets.Image(value=source['raw_bytes'],
                                 layout=widgets.Layout(max_width='200px', max_height='150px'))
        if row['modality'] == 'audio':
            return Audio(data=source['raw_bytes'])
        return None

    def on_search(_):
        query_out.clear_output()
        text = query_input.value.strip()
        if not text:
            with query_out:
                print("⚠️ Enter a query first.")
            return
        if client is None:
            with query_out:
                print("⚠️ No API client — run the Configuration cell first.")
            return

        with query_out:
            try:
                query_vec = _embed_query(text)
            except Exception as e:
                print(f"❌ Could not embed the query: {e}")
                return

            similarities = cosine_similarity([query_vec], embedding_matrix)[0]
            top_indices = np.argsort(similarities)[::-1][:topk_slider.value]

            for rank, idx in enumerate(top_indices, 1):
                row = embeddings_df.iloc[idx]
                cluster_note = ''
                if 'cluster' in row.index and row['cluster'] != -1:
                    cluster_note = f" · cluster {int(row['cluster'])}"

                header = (f'<p style="margin: 12px 0 2px 0;"><strong style="color: #274C77;">'
                          f'{rank}. {row["filename"]}</strong> '
                          f'<span style="color: #8B8C89;">({row["modality"]}, '
                          f'similarity {similarities[idx]:.3f}{cluster_note})</span></p>')
                display(HTML(header))

                if row.get('text_preview') and row['text_preview'] != 'N/A':
                    display(HTML(f'<p style="color: #274C77; margin: 2px 0 2px 15px; '
                                 f'font-size: 0.9em;">{row["text_preview"][:250]}</p>'))

                preview = _preview_widget(row)
                if preview is not None:
                    display(preview)

    search_btn.on_click(on_search)
    display(widgets.VBox([query_input, topk_slider, search_btn, query_out]))


## Export

Export your embedding results in multiple formats. Excel and CSV provide readable metadata. NumPy and Parquet store raw vectors efficiently. GEXF exports a similarity network for visualization in Gephi.

In [ ]:
# Export

if embedding_matrix is None:
    print("\u26a0\ufe0f Run the embedding cell first.")
else:
    export_format = widgets.Dropdown(
        options=[
            ('Excel (.xlsx) \u2014 metadata + similarity + clusters', 'excel'),
            ('CSV (.csv) \u2014 full data with embedding dimensions', 'csv'),
            ('JSON (.json) \u2014 full metadata and vectors', 'json'),
            ('NumPy (.npy) \u2014 raw embedding matrix', 'npy'),
            ('Parquet (.parquet) \u2014 columnar with embeddings', 'parquet'),
            ('GEXF (.gexf) \u2014 similarity network for Gephi', 'gexf'),
        ],
        value='excel',
        description='Format:',
        style={'description_width': '80px'},
        layout=widgets.Layout(width='500px')
    )

    # GEXF threshold (only shown for GEXF)
    gexf_threshold = widgets.FloatSlider(
        value=0.5, min=0.0, max=1.0, step=0.05,
        description='Similarity threshold:',
        style={'description_width': '160px'},
        layout=widgets.Layout(width='500px')
    )
    gexf_help = widgets.HTML(
        '<p style="color: #8B8C89; font-size: 0.85em; margin: 2px 0 10px 0;">'
        'Only edges above this threshold are included in the GEXF network. '
        'Lower values produce denser networks.</p>'
    )
    gexf_box = widgets.VBox([gexf_threshold, gexf_help])
    gexf_box.layout.display = 'none'

    def on_format_change(change):
        gexf_box.layout.display = '' if change['new'] == 'gexf' else 'none'
    export_format.observe(on_format_change, names='value')

    export_btn = widgets.Button(
        description='Export',
        style={'button_color': '#6096BA'},
        layout=widgets.Layout(width='200px', margin='10px 0')
    )

    export_out = widgets.Output()

    def on_export(_):
        export_out.clear_output()

        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        slug = make_slug(config.PROJECT_NAME) if config.PROJECT_NAME else 'project'
        fmt = export_format.value
        filepath = None

        try:
            if fmt == 'excel':
                filepath = f'embeddings_{slug}_{timestamp}.xlsx'

                # Sheet 1: Item metadata (no raw vectors)
                meta_df = embeddings_df[['filename', 'modality', 'mime_type', 'file_size_kb', 'text_preview']].copy()
                if 'cluster' in embeddings_df.columns:
                    meta_df['cluster'] = embeddings_df['cluster']

                # Sheet 2: Similarity matrix
                sim = cosine_similarity(embedding_matrix)
                sim_export = pd.DataFrame(sim, index=embeddings_df['filename'], columns=embeddings_df['filename'])

                # Sheet 3: Cluster summary
                cluster_summary = None
                if 'cluster' in embeddings_df.columns:
                    rows = []
                    for label in sorted(embeddings_df['cluster'].unique()):
                        members = embeddings_df[embeddings_df['cluster'] == label]
                        rows.append({
                            'Cluster': f'Cluster {label}' if label != -1 else 'Noise',
                            'Count': len(members),
                            'Files': ', '.join(members['filename'].tolist()),
                            'Modalities': ', '.join(f'{m}: {c}' for m, c in members['modality'].value_counts().items()),
                        })
                    cluster_summary = pd.DataFrame(rows)

                with pd.ExcelWriter(filepath, engine='openpyxl') as writer:
                    meta_df.to_excel(writer, sheet_name='Items', index=False)
                    sim_export.to_excel(writer, sheet_name='Similarity Matrix')
                    if cluster_summary is not None:
                        cluster_summary.to_excel(writer, sheet_name='Clusters', index=False)

                style_excel(filepath)

            elif fmt == 'csv':
                filepath = f'embeddings_{slug}_{timestamp}.csv'
                export_df = embeddings_df[['filename', 'modality', 'mime_type', 'file_size_kb', 'text_preview']].copy()
                if 'cluster' in embeddings_df.columns:
                    export_df['cluster'] = embeddings_df['cluster']
                # Add embedding dimensions as columns
                dim_cols = pd.DataFrame(
                    embedding_matrix,
                    columns=[f'dim_{i}' for i in range(embedding_matrix.shape[1])]
                )
                export_df = pd.concat([export_df.reset_index(drop=True), dim_cols], axis=1)
                export_df.to_csv(filepath, index=False)

            elif fmt == 'json':
                filepath = f'embeddings_{slug}_{timestamp}.json'
                output = {
                    'metadata': {
                        'project_name': config.PROJECT_NAME,
                        'project_description': config.PROJECT_DESCRIPTION,
                        'model': 'gemini-embedding-2-preview',
                        'dimensions': config.DIMENSIONS,
                        'task_type': config.TASK_TYPE or 'model default',
                        'n_items': len(embeddings_df),
                        'exported_at': datetime.now().isoformat(),
                    },
                    'items': [],
                }
                for _, row in embeddings_df.iterrows():
                    item = {
                        'filename': row['filename'],
                        'modality': row['modality'],
                        'mime_type': row['mime_type'],
                        'file_size_kb': row['file_size_kb'],
                        'text_preview': normalize_text(row['text_preview']) if row['text_preview'] != 'N/A' else None,
                        'embedding': row['embedding'],
                    }
                    if 'cluster' in row.index:
                        item['cluster'] = int(row['cluster'])
                    output['items'].append(item)

                with open(filepath, 'w') as f:
                    json.dump(output, f, indent=2, default=str)

            elif fmt == 'npy':
                filepath = f'embeddings_{slug}_{timestamp}.npy'
                np.save(filepath, embedding_matrix)

                # Save companion metadata JSON
                meta_path = f'embeddings_{slug}_{timestamp}_metadata.json'
                meta = {
                    'project_name': config.PROJECT_NAME,
                    'dimensions': config.DIMENSIONS,
                    'task_type': config.TASK_TYPE or 'model default',
                    'shape': list(embedding_matrix.shape),
                    'files': embeddings_df['filename'].tolist(),
                    'modalities': embeddings_df['modality'].tolist(),
                }
                if 'cluster' in embeddings_df.columns:
                    meta['clusters'] = embeddings_df['cluster'].tolist()
                with open(meta_path, 'w') as f:
                    json.dump(meta, f, indent=2)

                with export_out:
                    print(f"\u2713 NumPy: {filepath}")
                    print(f"\u2713 Metadata: {meta_path}")
                    if IN_COLAB:
                        colab_files.download(filepath)
                        colab_files.download(meta_path)
                return  # Early return - two files

            elif fmt == 'parquet':
                filepath = f'embeddings_{slug}_{timestamp}.parquet'
                export_df = embeddings_df[['filename', 'modality', 'mime_type', 'file_size_kb', 'text_preview']].copy()
                if 'cluster' in embeddings_df.columns:
                    export_df['cluster'] = embeddings_df['cluster']
                dim_cols = pd.DataFrame(
                    embedding_matrix,
                    columns=[f'dim_{i}' for i in range(embedding_matrix.shape[1])]
                )
                export_df = pd.concat([export_df.reset_index(drop=True), dim_cols], axis=1)
                export_df.to_parquet(filepath, index=False)

            elif fmt == 'gexf':
                if not HAVE_NETWORKX:
                    with export_out:
                        print("\u26a0\ufe0f networkx is not installed \u2014 GEXF export unavailable.")
                    return

                filepath = f'similarity_network_{slug}_{timestamp}.gexf'
                threshold = gexf_threshold.value

                G = nx.Graph()
                for idx, row in embeddings_df.iterrows():
                    attrs = {'modality': row['modality'], 'file_size_kb': row['file_size_kb']}
                    if 'cluster' in row.index:
                        attrs['cluster'] = int(row['cluster'])
                    G.add_node(row['filename'], **attrs)

                sim = cosine_similarity(embedding_matrix)
                n = len(sim)
                edge_count = 0
                for i in range(n):
                    for j in range(i + 1, n):
                        if sim[i, j] >= threshold:
                            G.add_edge(
                                embeddings_df.iloc[i]['filename'],
                                embeddings_df.iloc[j]['filename'],
                                weight=round(float(sim[i, j]), 4)
                            )
                            edge_count += 1

                nx.write_gexf(G, filepath)

                with export_out:
                    print(f"\u2713 GEXF: {filepath}")
                    print(f"   Nodes: {G.number_of_nodes()}, Edges: {edge_count} (threshold: {threshold})")
                    if IN_COLAB:
                        colab_files.download(filepath)
                return  # Early return - custom output

            # Standard single-file output
            with export_out:
                print(f"\u2713 Saved: {filepath}")
                if IN_COLAB:
                    colab_files.download(filepath)

        except Exception as e:
            with export_out:
                print(f"\u274c Export error: {type(e).__name__}: {e}")

    export_btn.on_click(on_export)

    display(widgets.VBox([
        export_format,
        gexf_box,
        export_btn,
        export_out,
    ]))